# Python Package: `Refspy`

A Python package for indexing biblical references.

- [pypi.org/eukras/refspy](https://pypi.org/project/refspy/)
- [refspy.readthedocs.io](https://refspy.readthedocs.io/en/latest/refspy.html)

## Initialisation

In [1]:
from refspy import refspy

__ = refspy()                     #  <-- Refspy defaults to a Protestant canon and en_US localisation,
__ = refspy("orthodox")           #  <-- but we'll include deuterocanonical books for this demonstration.
FR = refspy('orthodox', 'fr_FR')  #  <-- French localisation. 

## Formatting 

In [2]:
r = __.r("Gen 1:1")

print([__.name(r), __.book(r), __.abbrev_name(r), __.abbrev_book(r), __.numbers(r)])
print([FR.name(r), FR.book(r), FR.abbrev_name(r), FR.abbrev_book(r), FR.numbers(r)])



['Genesis 1:1', 'Genesis', 'Gen 1:1', 'Gen', '1:1']
['Genèse 1, 1', 'Genèse', 'Gn 1, 1', 'Gn', '1, 1']


## Find references

Some demonstration text is stored in each language file.

In [3]:
from refspy.languages.english import ENGLISH

print(ENGLISH.demonstration_text)

Human-written Bible references look like Rom 12.1-2, 9-12, 2 Cor. 4:16-5:5,
or Philemon 4-7. They wrap lines, use spaces inconsistently, use colons and
periods interchangeably, indicate abbreviations with periods (or not), and
have commas both between and within references. They are sometimes
malformed, like Mt 1.10000 or 1:3-2, but might also use number
abbreviations like Ps 119:105-12. They might refer to Deuterocanonical
books (DC), like Wis 7:21-30 or 2 Macc 7, or Anagignoskomena (DCO), like 1
Esdras 4:35-40. A book name, like Second Corinthians, may provide context
for references that follow, such as 5:11-15 or vv. 16-21. We want to match,
say, 'John' but not match 'John Smith' in these cases. If we cite Romans
(but then add a reference to 2ndCo 5:11-21 in parentheses), a subsequent
reference like 12:9-21 should still be to Romans. Using letters for partial
verses, as in II Cor 5:11a-15d, has no consistent meaning, so the letters
are ignored. Book aliases that are common words wil

In [4]:
text = ENGLISH.demonstration_text

matches = __.find_references(text, include_books=True, include_nones=True)

columns = "%-10s %-30s %-30s %-30s"
print(columns % ("REF?", "MATCH", "NAME", "ABBREV_NAME"))
print(columns % ("----", "-----", "----", "-----------"))

for match, ref in matches:
    if ref is None:
        print(columns % ("None", '"' + match.replace('\n', '<CR>') + '"', '--', '--'))
    elif ref.is_book():
        print(columns % ("Book", '"' + match.replace('\n', '<CR>') + '"', __.name(ref), __.abbrev_name(ref)))
    else:
        print(columns % ("", '"' + match.replace('\n', '<CR>') + '"', __.name(ref), __.abbrev_name(ref)))

REF?       MATCH                          NAME                           ABBREV_NAME                   
----       -----                          ----                           -----------                   
           "Rom 12.1-2, 9-12"             Romans 12:1–2, 9–12            Rom 12:1–2, 9–12              
           "2 Cor. 4:16-5:5"              2 Corinthians 4:16–5:5         2 Cor 4:16–5:5                
           "Philemon 4-7"                 Philemon 4–7                   Phlm 4–7                      
None       "Mt 1.10000"                   --                             --                            
None       "1:3-2"                        --                             --                            
           "Ps 119:105-12"                Psalm 119:105–112              Ps 119:105–112                
           "Wis 7:21-30"                  Wisdom of Solomon 7:21–30      Wis 7:21–30                   
           "2 Macc 7"                     2 Maccabees 7         

## Sort and compare

References can be compared directly using comparison operators. 

In [5]:
if __.r('Matt 4') < __.r('Luke 7'):
    print('Matt 4 < Luke 7')

Matt 4 < Luke 7


This of course means they can be `sorted()`. 

In [6]:
sorted_refs = sorted([ref for match_str, ref in matches if ref and not ref.is_book()])
formatted_refs = [__.name(ref) for ref in sorted_refs]
for i, name in enumerate(formatted_refs):
    print(f"{i + 1}. {name}")

1. Psalm 119:105–112
2. Amos 4:1
3. Wisdom of Solomon 7:21–30
4. 2 Maccabees 7
5. 1 Esdras 4:35–40
6. Romans 12:1–2, 9–12
7. Romans 12:9–21
8. 2 Corinthians 4:16–5:5
9. 2 Corinthians 5:11–15
10. 2 Corinthians 5:11–15
11. 2 Corinthians 5:11–21
12. 2 Corinthians 5:16–21
13. Philemon 4–7


## Collate References

In [7]:
references = [ref for ref in sorted_refs if not ref.is_book()]

for library, book_collation in __.collate_chapter_references(references):
    print(f"{library.name}")
    for book, chapter_collation in book_collation:
        print(f"    {book.name}")
        for _, chapter_references in chapter_collation:
            print(f"       ", __.numbers(__.join(chapter_references)))

Old Testament
    Psalm
        119:105–112
    Amos
        4:1
Deuterocanonical
    Wisdom of Solomon
        7:21–30
    2 Maccabees
        7
Deuterocanonical (Orthodox)
    1 Esdras
        4:35–40
New Testament
    Romans
        12:1–2, 9–12, 9–21
    2 Corinthians
        4:16–5:5
        5:11–15, 11–15, 11–21, 16–21
    Philemon
        4–7


## Generate an index

There are shortcuts for common indexing tasks. 

In [8]:
references = [ref for _, ref in matches if ref and not ref.is_book()]

print(__.make_index(references))
print(__.make_summary(references))  # <-- merge overlapping and combine adjacent references 
print(__.make_hotspots(references))

Ps 119:105–112; Amos 4:1; Wis 7:21–30; 2 Macc 7; 1 Esd 4:35–40; Rom 12:1–2, 9–12, 9–21; 2 Cor 4:16–5:5; 5:11–15, 11–15, 11–21, 16–21; Phlm 4–7
Ps 119:105–112; Amos 4:1; Wis 7:21–30; 2 Macc 7; 1 Esd 4:35–40; Rom 12:1–2, 9–21; 2 Cor 4:16–5:5; 5:11–21; Phlm 4–7
2 Cor 5, Rom 12


## Add HTML links to references

To link references to online Bibles, just use `__.template()` with the template variables listed in `README.md`. 

In [9]:
bible_gateway = (
    '<a href="https://www.biblegateway.com/passage/'
  + '?search={LINK}&version=NRSVA">'
  + '{ABBREV_NAME}'
  + '</a>'
)
ref = __.r('2 Cor 3:4-5')  # <-- note this shorthand way to create a reference

from IPython.display import HTML  #<-- for showing HTML in this notebook

HTML("<ul><li>" + __.template(ref, bible_gateway) + "</li></ul>")


## Replace references with HTML links

In [12]:
from refspy.utils import sequential_replace

strs, tags = [], []
for match_str, ref in __.find_references(text,
                                         include_books=True,
                                         include_nones=True):
    strs.append(match_str)
    if ref is None: 
          tags.append(f'<span style="color: red;">{match_str}</span>')
    elif ref.is_book():
        tags.append(f'<span style="color: green;">{match_str}</span>')
    else:
        tags.append(
             f'<b>{match_str}</b> <sup>[{__.template(ref, bible_gateway)}]</sup>'
        )

html = sequential_replace(text, strs, tags)

HTML(f"<output style=\"font-size: larger;\">{html}</output>")